# RunPod Models by Benchmark Type

Run selected local Hugging Face vision models against selected benchmark groups on a RunPod GPU. First run `setup_runpod.ipynb`, then open this notebook from `/workspace/transformers`. This notebook does not clone or update the repository. It follows the structure of `api_models.ipynb`, but loads one model at a time to limit VRAM use.

Every completed model/benchmark pair is written to its own JSON file. `run_summary.json` is updated atomically after every success, skip, or failure, so interrupted runs can be resumed. For persistent storage, set `RUNPOD_RESULTS_DIR` to a directory on the mounted RunPod volume before starting the notebook.

## 1. Setup

In [ ]:
from pathlib import Path
import gc
import getpass
import importlib
import inspect
import json
import os
import sys
import traceback

# RunPod mounts persistent pod or network storage at /workspace.
# Saving results there preserves them across pod stops and restarts.
if Path("/workspace").is_dir():
    os.environ.setdefault("RUNPOD_RESULTS_DIR", "/workspace/results")

candidate_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (path for path in candidate_roots if (path / "models").is_dir() and (path / "benchmarks").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        "This notebook must be opened from the cloned transformers repository. "
        "Run setup_runpod.ipynb first, then open "
        "/workspace/transformers/runpod_models.ipynb."
    )

REPO_ROOT = REPO_ROOT.resolve()
os.chdir(REPO_ROOT)
requirements_path = REPO_ROOT / "requirements.txt"
%pip install -q -r "$requirements_path"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import torch
from huggingface_hub import login

from benchmark_groups import BENCHMARK_TYPE_TITLES, selected_benchmark_specs
from runners.benchmark_run import BenchmarkRun
from runners.execution import run_benchmark_matrix
from runners.model_run import ModelRun

HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
if not HF_TOKEN:
    entered_token = getpass.getpass("HF token (press Enter to continue without one): ").strip()
    if entered_token:
        HF_TOKEN = entered_token
        os.environ["HF_TOKEN"] = entered_token
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU was detected. Start this notebook in a GPU-backed RunPod pod.")

print(f"Repository: {REPO_ROOT}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"HF authentication: {'configured' if HF_TOKEN else 'not configured'}")

## 2. Choose Models and Benchmarks

Models are configured as import paths so they are instantiated only when their turn starts. The list below is the single-B300 RunPod set from the open-weight execution plan in `docs/tex/Report.tex`. `mistral-medium-3.5` is marked borderline because its estimated weight-only footprint is close to the B300's capacity. `SKIP_EXISTING=True` resumes a previous run without replacing completed pair results.

In [ ]:
RUN_NAME = "runpod_models"
RESULTS_ROOT = Path(os.getenv("RUNPOD_RESULTS_DIR", REPO_ROOT / "results")).expanduser()
OUTPUT_DIR = RESULTS_ROOT / RUN_NAME
SUMMARY_PATH = OUTPUT_DIR / "run_summary.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_SAMPLES = 20
STREAMING = True
SKIP_EXISTING = True

MODEL_SPECS = [
    {
        "name": "dots.ocr",
        "module": "models.dots_ocr",
        "class": "DotsOCR",
        "kwargs": {"temperature": 0.0},
        "weight_only_vram_gb": 5.8,
    },
    {
        "name": "glm-4.1v-thinking",
        "module": "models.glm_4_1v_thinking",
        "class": "GLM41VThinking",
        "kwargs": {"temperature": 0.0},
        "weight_only_vram_gb": 18,
    },
    {
        "name": "gemma-4-26b-a4b",
        "module": "models.gemma4_26b_a4b_it",
        "class": "Gemma4_26BA4B",
        "kwargs": {"temperature": 0.0},
        "weight_only_vram_gb": 52,
    },
    {
        "name": "qwen3.6-27b",
        "module": "models.qwen3_6_27b",
        "class": "Qwen36_27B",
        "kwargs": {"temperature": 0.0},
        "weight_only_vram_gb": 54,
    },
    {
        "name": "gemma-4-31b",
        "module": "models.gemma4_31b_it",
        "class": "Gemma4_31B",
        "kwargs": {"temperature": 0.0},
        "weight_only_vram_gb": 62,
    },
    {
        "name": "skywork-r1v2-38b",
        "module": "models.skywork_r1v2_38b",
        "class": "SkyworkR1V2_38B",
        "kwargs": {"temperature": 0.0},
        "weight_only_vram_gb": 76,
    },
    {
        "name": "skywork-r1v3-38b",
        "module": "models.skywork_r1v3_38b",
        "class": "SkyworkR1V3_38B",
        "kwargs": {"temperature": 0.0},
        "weight_only_vram_gb": 76,
    },
    {
        "name": "mistral-small-4",
        "module": "models.mistral_small_4",
        "class": "MistralSmall4",
        "kwargs": {"temperature": 0.0},
        "weight_only_vram_gb": 119,
    },
    {
        "name": "glm-4.5v-thinking",
        "module": "models.glm_4_5v_thinking",
        "class": "GLM45VThinking",
        "kwargs": {"temperature": 0.0},
        "weight_only_vram_gb": 212,
    },
    {
        "name": "mistral-medium-3.5",
        "module": "models.mistral_medium_3_5",
        "class": "MistralMedium35",
        "kwargs": {"temperature": 0.0},
        "weight_only_vram_gb": 256,
        "borderline": True,
    },
]

SELECTED_BENCHMARKS_BY_TYPE = {
    "A": "ALL",
    "B": "ALL",
    "C": "ALL",
    "E": "ALL",
    "G": "ALL",
    "L": "ALL",
    "P": "ALL",
    "R": "ALL",
    # "A": ["VQA v2.0", "GQA"],
    # "C": ["Flickr30k", "MS COCO Captions"],
}

selected_specs = selected_benchmark_specs(SELECTED_BENCHMARKS_BY_TYPE)
print(f"Results: {OUTPUT_DIR}")
model_plan_df = pd.DataFrame(MODEL_SPECS)[
    ["name", "module", "class", "weight_only_vram_gb"]
].copy()
model_plan_df["borderline"] = [spec.get("borderline", False) for spec in MODEL_SPECS]
display(model_plan_df)
print(f"Models: {[spec['name'] for spec in MODEL_SPECS]}")
print(f"Matrix pairs: {len(MODEL_SPECS) * len(selected_specs)}")
pd.DataFrame(selected_specs)[["type", "type_title", "name", "module", "class"]]

## 3. Run Matrix

In [ ]:
def load_class(module_name, class_name):
    module = importlib.import_module(module_name)
    return getattr(module, class_name)


def supported_kwargs(model_class, configured_kwargs):
    parameters = inspect.signature(model_class).parameters
    return {key: value for key, value in configured_kwargs.items() if key in parameters}


def write_summary(entries):
    temporary_path = SUMMARY_PATH.with_suffix(".json.tmp")
    temporary_path.write_text(
        json.dumps(entries, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary_path.replace(SUMMARY_PATH)


summaries = []
if SUMMARY_PATH.exists():
    try:
        summaries = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        print(f"[WARN] Could not read existing summary: {SUMMARY_PATH}")

latest_by_pair = {
    (entry.get("model"), entry.get("benchmark")): entry
    for entry in summaries
}

for model_spec in MODEL_SPECS:
    model_name = model_spec["name"]
    model_run = None
    try:
        model_class = load_class(model_spec["module"], model_spec["class"])
        kwargs = supported_kwargs(model_class, model_spec.get("kwargs", {}))
        print(f"\n=== Loading {model_name} ===", flush=True)
        model_run = ModelRun.from_factory(model_name, model_class, **kwargs)

        for spec in selected_specs:
            benchmark_cls = load_class(spec["module"], spec["class"])
            benchmark_name = benchmark_cls.benchmark_name or spec["name"]
            result_path = OUTPUT_DIR / f"{model_name}_{benchmark_name}.json"
            pair_key = (model_name, benchmark_name)

            if SKIP_EXISTING and result_path.exists():
                entry = {
                    "type": spec["type"],
                    "type_title": spec["type_title"],
                    "model": model_name,
                    "benchmark": benchmark_name,
                    "status": "skipped",
                    "results_path": str(result_path),
                }
                print(f"[SKIP] {model_name} / {benchmark_name}", flush=True)
            else:
                print(
                    f"[RUN] Type {spec['type']}: {BENCHMARK_TYPE_TITLES[spec['type']]} / "
                    f"{model_name} / {benchmark_name}",
                    flush=True,
                )
                try:
                    benchmark = benchmark_cls(streaming=STREAMING)
                    model_run.model.max_new_tokens = benchmark.default_max_new_tokens
                    sample_count = min(NUM_SAMPLES, 9) if benchmark.name == "ucf101" else NUM_SAMPLES
                    row = run_benchmark_matrix(
                        models=[model_run],
                        benchmark_runs=[BenchmarkRun(benchmark=benchmark, num_samples=sample_count)],
                        output_dir=OUTPUT_DIR,
                    )[0]
                    entry = {
                        "type": spec["type"],
                        "type_title": spec["type_title"],
                        "status": "ok",
                        **row,
                    }
                except Exception as exc:
                    entry = {
                        "type": spec["type"],
                        "type_title": spec["type_title"],
                        "model": model_name,
                        "benchmark": benchmark_name,
                        "status": "error",
                        "error": f"{type(exc).__name__}: {exc}",
                        "traceback": traceback.format_exc(),
                    }

                print(f"[{entry['status'].upper()}] {model_name} / {benchmark_name}", flush=True)

            latest_by_pair[pair_key] = entry
            summaries = list(latest_by_pair.values())
            write_summary(summaries)
    except Exception as exc:
        print(f"[MODEL ERROR] {model_name}: {type(exc).__name__}: {exc}", flush=True)
        for spec in selected_specs:
            benchmark_cls = load_class(spec["module"], spec["class"])
            benchmark_name = benchmark_cls.benchmark_name or spec["name"]
            pair_key = (model_name, benchmark_name)
            if SKIP_EXISTING and (OUTPUT_DIR / f"{model_name}_{benchmark_name}.json").exists():
                continue
            latest_by_pair[pair_key] = {
                "type": spec["type"],
                "type_title": spec["type_title"],
                "model": model_name,
                "benchmark": benchmark_name,
                "status": "model_error",
                "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc(),
            }
        summaries = list(latest_by_pair.values())
        write_summary(summaries)
    finally:
        if model_run is not None:
            del model_run
        gc.collect()
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(summaries)
display(summary_df)
print(f"Summary saved to: {SUMMARY_PATH}")

## 4. Inspect Saved Results

In [ ]:
summary_df = pd.DataFrame(json.loads(SUMMARY_PATH.read_text(encoding="utf-8")))
display(summary_df.groupby(["type", "type_title", "status"]).size().rename("count").reset_index())

sample_rows = []
for summary in summary_df.to_dict("records"):
    path = summary.get("results_path")
    if not path or not Path(path).exists():
        continue
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    report = payload.get("report", {})
    for result in report.get("results", []):
        sample_rows.append({
            "type": summary["type"],
            "model": summary["model"],
            "benchmark": summary["benchmark"],
            "sample_index": result.get("index"),
            "total": result.get("total"),
            "correct": result.get("correct"),
            "prediction": result.get("prediction"),
            "error": result.get("error") or result.get("stats", {}).get("error"),
        })

sample_df = pd.DataFrame(sample_rows)
sample_df